<img src="https://www.epfl.ch/about/overview/wp-content/uploads/2020/07/logo-epfl-1024x576.png" width="140px" alt="EPFL_logo">

## Image Processing Laboratory Notebooks
---

This Jupyter Notebook is part of a series of computer laboratories that are designed
to teach image-processing programming; they are running on the EPFL's Noto server. They are the practical complement of the theoretical lectures of the EPFL's Master course 
[**MICRO-512 Image Processing II**](https://moodle.epfl.ch/course/view.php?id=463) taught by Prof. M. Unser and Prof. D. Van de Ville.

The project is funded by the Center for Digital Education and the School of Engineering. It is owned by the [Biomedical Imaging Group](http://bigwww.epfl.ch/). 
The distribution or reproduction of the notebook is strictly prohibited without the written consent of the authors.  &copy; EPFL 2026.

**Authors**: 
    [Pol del Aguila Pla](mailto:pol.delaguilapla@epfl.ch), 
    [Kay Lächler](mailto:kay.lachler@epfl.ch),
    [Alejandro Noguerón Arámburu](mailto:alejandro.nogueronaramburu@epfl.ch), and
    [Daniel Sage](mailto:daniel.sage@epfl.ch).
    
To ensure your work is graded correctly by our automated system, **do not create new cells or delete/rearrange/copy existing ones** when you submit. The current cells contain hidden metadata required for the auto-grader to identify your solutions. If you create temporary cells for testing during your work, remember to clean them up before submission.

# Lab 4.2: Orientation
**Released**: Thursday, February 19, 2026

**Submission deadline**: Monday, March 2, 2026, before 23:59 on [Moodle](https://moodle.epfl.ch/course/view.php?id=463)

**Grade weight**: Lab 4 (18 points), 7.5 % of the overall grade

**Help Session**: Thursday, February 26, 2026

**Related lectures**: Chapter 6

### Student Name: 
### SCIPER: 

Double-click on this cell and fill your name and SCIPER number. Then, run the cell below to verify your identity in Noto and set the seed for random results.

In [ ]:
import getpass
# This line recovers your camipro number to mark the images with your ID
uid = int(getpass.getuser().split('-')[2]) if len(getpass.getuser().split('-')) > 2 else ord(getpass.getuser()[0])
print(f'SCIPER: {uid}')

## Imports
In the next cell, we import Python libraries we will use throughout the lab, as well as the `IPLabViewer` class, created specifically for this course, which provides interactive image visualization based on the `ipywidgets` library:
* [`matplotlib.pyplot`](https://matplotlib.org/3.2.2/api/_as_gen/matplotlib.pyplot.html), to display images
* [`ipywidgets`](https://ipywidgets.readthedocs.io/en/latest/), to make the image display interactive
* [`numpy`](https://numpy.org/doc/stable/reference/index.html), for mathematical operations on arrays
* [`openCV` (cv2)](https://docs.opencv.org/2.4/index.html), for image-processing tasks",
* [`scikit-image` (skimage)](https://scikit-image.org/docs/stable/api/api.html), also for image-processing tasks",

We will then load the `ImageViewer` class (see the documentation [here](https://github.com/Biomedical-Imaging-Group/interactive-kit/wiki/Image-Viewer), or run the Python command `help(viewer)` after loading the class).

Finally, we load the images you will use in the exercise to test your functions. 

In [ ]:
# Configure plotting as dynamic
%matplotlib widget

# Import standard required packages for this exercise
import warnings
import matplotlib.pyplot as plt
import ipywidgets as widgets
import numpy as np
import cv2 as cv 
import skimage
from skimage import feature
from interactive_kit import imviewer as viewer 

# Load images to be used in this exercise
wave_ramp        = cv.imread('images/wave-ramp.tif',       cv.IMREAD_UNCHANGED)
wave_ramp        = (255*(wave_ramp-wave_ramp.min())/(wave_ramp.max()-wave_ramp.min())).astype(np.uint8)
fingerprint      = cv.imread('images/fingerprint.tif',     cv.IMREAD_UNCHANGED)
turbulence       = cv.imread('images/turbulence.tif',      cv.IMREAD_UNCHANGED).T
traces           = cv.imread('images/traces.tif',          cv.IMREAD_UNCHANGED)
fourierhouse    = cv.imread('images/fourierhouse.tif',   cv.IMREAD_UNCHANGED)

# Orientation laboratory (12 points)

In this lab we will implement the computation of the structure tensor presented in Chapter 6.2, which can be used to perform directional image analysis.

The block-diagram of the complete system is shown in the following flowchart, where $f(x,y)$ is the graylevel input image.

<center><img src="images/block-diagram.png" alt="Drawing" style="width: 800px;"/></center>

Successively, you will implement the functions
* `structure_tensor` to generate the structure tensor matrix,
* `orientation_features`, which implements the whole chain of calculations to generate the features needed for directional analysis, and
* `colorize_features` to display the calculated features as a color image.


Once you ensured that the functions are correct, you will use them in two applications that rely on directional image analysis, 
* a method to select specific orientations, and
* a vector field generator. 

Finally, we will analyze the `structure_tensor` function and see if we can improve the obtained results.

**Note:** This part of the lab will be carried out completely in Python.

### Visualize images
First of all, run the cell below to get familiar with the images we will be using. Remember you can use `Next` and `Prev` to cycle through the images.

In [ ]:
# Declare image_list for ImageViewer
plt.close('all')
img_list = [ fingerprint, wave_ramp, turbulence, traces, fourierhouse]
imgs_viewer = viewer(img_list, widgets=True)

## Structure tensor matrix (2 points)

To calculate the elements $J_{xx}$, $J_{xy}$ and $J_{yy}$ of the structure tensor, we only need two computational blocks: a Gaussian filter and a gradient filter. For the gradient, we will use the **OpenCV Sobel filter** [`cv.Sobel(src, ddepth, dx, dy, ksize)`](https://docs.opencv.org/3.4/d4/d86/group__imgproc__filter.html#gacea54f142e81b6758cb6f375ce782c8d). Click on the link to see its documentation and choose the right input parameters to apply a **first-order** Sobel filter of **size $3\times3$**. 

For the smoothing filter, we will use the standard Gaussian filter provided by OpenCV [`cv.GaussianBlur(fxx, ksize, sigmaX)`](https://docs.opencv.org/master/d4/d86/group__imgproc__filter.html#gaabe8c836e97159a9193fb0b11ac52cf1) with default boundary conditions (`BORDER_DEFAULT` or `BORDER_REFLECT_101`). If you are unsure about its input parameters, go check the documentation.

**For 2 points**, complete the function `structure_tensor` in the cell below. It may be useful to revisit the [figure](#Orientation-laboratory-(12-points)) at the start of this notebook before starting.

**Hints:** 
* **`cv.Sobel`:** set `ddepth=cv.CV_64F` since the gradient can be a negative floating point number.
* **`cv.GaussianBlur`:** set `ksize=(0,0)` to let the function choose the filter size automatically from `sigmaX`.
* **Multiplication:** Note that, contrary to Matlab and other languages, the Python operator `*` applied to a NumPy Array performs *element-wise* multiplication.
* To ensure that you define the right parameter in a function call, always specify the name of the parameter. For example: `cv.Sobel(src=img, ddepth=cv.CV_64F, dx=1, dy=1, ksize=5)` is much easier to understand and debug than `cv.Sobel(img, cv.CV_64F, 1, 1, 5)`.

In [ ]:
# Function that calculates the elements Jxx, Jxy and Jyy of the structure tensor matrix
def structure_tensor(img, sigma):
    Jxx = np.zeros(img.shape)
    Jxy = np.zeros(img.shape)
    Jyy = np.zeros(img.shape)
    
    # YOUR CODE HERE
    
    return Jxx, Jxy, Jyy

Let's perform a quick sanity check on a simple $11 \times 11$ **impulse image** using `sigma=1`. 

**Note:** You can modify the input image and the sigma value in the cell below to observe the different results.

In [ ]:
# Define impulse image
size = 11
test_img = np.zeros((size,size))
test_img[size//2, size//2] = 1
# Run function and display the result
Jxx, Jxy, Jyy = structure_tensor(test_img, sigma=1)
plt.close('all')
view = viewer([test_img, Jxx, Jyy, Jxy], subplots=(2,2))

Specifically, we will first check that $J_{xx}$ and $J_{yy}$ are non-negative and identical to each other when rotated by $90^{\circ}$, which should be the case for this impulse image. Then, we will also check that $J_{xy}$ contains both negative and non-negative numbers, and that all elements that are either in the fifth row or the fifth column of $J_{xy}$ are zero. 

Because the structure tensor is a crucial part of this lab, we will also perform more sophisticated sanity checks comparing your results to our pre-computed correct results.

In [ ]:
# Basic sanity checks
# Re-run example in case you've played around with the previous cell
test_img = np.zeros((11,11)); test_img[5, 5] = 1
Jxx, Jxy, Jyy = structure_tensor(test_img, sigma = 1)

# No negative values in Jxx and Jyy
if not (np.all(Jxx >= 0) and np.all(Jyy >= 0)):
    print('WARNING!\nJxx and Jyy should not contain any negative values.\n')
if not np.allclose(np.rot90(Jxx), Jyy):
    print('WARNING!\nJxx should be the same as Jyy but rotated 90 degrees.\n')

# Jxy both positive and negative
if not (np.any(Jxy > 0) and np.any(Jxy < 0)):
    print('WARNING!\nJxy should contain both negative and positive values.\n')

# Fifth row/col zeros
if not (np.all(abs(Jxy[5, :]) < 1e-5) and np.all(abs(Jxy[:, 5]) < 1e-5)):
    print('WARNING!\nThe fifth row/column of Jxy should only have zeros.\n')

# Comparison to pre-computed correct results
# Boundaries should be close to zero
error_check = [False, False, False]
mask = np.ones(test_img.shape, dtype=bool); mask[2:9, 2:9] = False
if not np.all(np.abs(Jxx[mask]) < 0.01):
    print('WARNING!\nJxx is not yet correct, values outside the range x=[2,8], y=[2,8] should be close to 0.\n')
    error_check[0] = True
if not np.all(np.abs(Jyy[mask]) < 0.01):
    print('WARNING!\nJyy is not yet correct, values outside the range x=[2,8], y=[2,8] should be close to 0.\n')
    error_check[1] = True
if not np.all(np.abs(Jxy[mask]) < 0.01):
    print('WARNING!\nJxy is not yet correct, values outside the range x=[2,8], y=[2,8] should be close to 0.\n')
    error_check[2] = True
# Correct outputs
Jxx_corr = np.array([[0.004, 0.018, 0.033, 0.035, 0.033, 0.018, 0.004],
                     [0.025, 0.114, 0.209, 0.224, 0.209, 0.114, 0.025],
                     [0.077, 0.35,  0.644, 0.688, 0.644, 0.35,  0.077],
                     [0.113, 0.512, 0.942, 1.006, 0.942, 0.512, 0.113],
                     [0.077, 0.35,  0.644, 0.688, 0.644, 0.35,  0.077],
                     [0.025, 0.114, 0.209, 0.224, 0.209, 0.114, 0.025],
                     [0.004, 0.018, 0.033, 0.035, 0.033, 0.018, 0.004]])
Jyy_corr = Jxx_corr.T
Jxy_corr = np.array([[ 0.003,  0.013,  0.019,  0.   , -0.019, -0.013, -0.003],
                     [ 0.013,  0.056,  0.082,  0.   , -0.082, -0.056, -0.013],
                     [ 0.019,  0.082,  0.119,  0.   , -0.119, -0.082, -0.019],
                     [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
                     [-0.019, -0.082, -0.119,  0.   ,  0.119,  0.082,  0.019],
                     [-0.013, -0.056, -0.082,  0.   ,  0.082,  0.056,  0.013],
                     [-0.003, -0.013, -0.019,  0.   ,  0.019,  0.013,  0.003]])

# Exact values in the center
if not np.allclose(Jxx[2:9, 2:9], Jxx_corr, atol=1e-3):
    print('WARNING!\nThe non-zero values of Jxx inside the range x=[2,8], y=[2,8] are not yet correct.')
    error_check[0] = True
if not np.allclose(Jyy[2:9, 2:9], Jyy_corr, atol=1e-3):
    print('WARNING!\nThe non-zero values of Jyy inside the range x=[2,8], y=[2,8] are not yet correct.')
    error_check[1] = True
if not np.allclose(Jxy[2:9, 2:9], Jxy_corr, atol=1e-3):
    print('WARNING!\nThe non-zero values of Jxy inside the range x=[2,8], y=[2,8] are not yet correct.')
    error_check[2] = True

# In the presence of errors, show differences to debug your code
if np.any(error_check):
    print('\nLook at the output below to compare your solution to the correct one:\n')
    Jxx_vis = np.zeros(test_img.shape); Jyy_vis = np.zeros(test_img.shape); Jxy_vis = np.zeros(test_img.shape)
    Jxx_vis[mask == False] = Jxx_corr.flatten(); Jyy_vis[mask == False] = Jyy_corr.flatten(); Jxy_vis[mask == False] = Jxy_corr.flatten()
    corr_imgs = [Jxx_vis, Jyy_vis, Jxy_vis]; imgs = [Jxx, Jyy, Jxy]; names = ['Jxx', 'Jyy', 'Jxy']
    img_list = []; title_list = []; err_count = 0
    for i, c in enumerate(error_check):
        if c:
            img_list.append(imgs[i]); img_list.append(corr_imgs[i])
            title_list.append(names[i]); title_list.append(names[i] + ' correct')
            err_count += 1
    plt.close('all'); view = viewer(img_list, title=title_list, subplots=(err_count,2))
else:
    print('Congratulations! Your structure_tensor passed the sanity check.')

Now you can also apply `structure_tensor` to the various images we have imported before (wave_ramp, fingerprint, turbulence, traces or fourierhouse) and see what the elements of the structure tensor matrix look like. Run the cell below and change the variable `image` to different images. You can also change the `sigma` value and observe the effect this has on the result.

In [ ]:
# you can change the image to any of the ones we imported: 
# fingerprint, wave_ramp, turbulence, traces or fourierhouse
image = fingerprint

Jxx, Jxy, Jyy = structure_tensor(image, sigma=2)
plt.close('all')
view = viewer([image, Jxx, Jxy, Jyy], subplots=(2,2))

## Orientation features

Now that we have the structure tensor, we can compute useful features derived from it to help us understand and visualize the orientation of an image. An easy way to think about the structure tensor is that, for each pixel location `[m,n]`, we can calculate a matrix $\mathbf{J}$ made out of the values of $J_{xx}$, $J_{yy}$, and $J_{xy}$ at that same pixel, i.e.,

$$
    \mathbf{J}[m,n] = \left[ \begin{array}{cc} J_{xx}[m,n] & J_{xy}[m,n] \\ J_{xy}[m,n] & J_{yy}[m,n]\end{array} \right]\,.
$$

### Feature calculation (4 points)

In the table below, you can see the four features that we are going to compute from this matrix $\mathbf{J}$, where we drop the pixel indeces `[m,n]` for simplicity. There, $\det(\mathbf{J})$ stands for the determinant of the matrix, and $\operatorname{tr}(\mathbf{J})$ for the trace.

| Feature | Relation to the structure tensor matrix $\mathbf{J}$ |
| :-: | :-: |
| Orientation | $$\Theta = \frac{1}{2}\arctan\left(\frac{2J_{xy}}{J_{yy}-J_{xx}}\right)$$ |
| Gradient Energy | $$E = J_{yy}+J_{xx}$$ |
| Coherence | $$C = \frac{\sqrt{(J_{yy}-J_{xx})^2+4J_{xy}^2}}{J_{yy}+J_{xx}}$$ |
| Harris Index | $$H = \det(\mathbf{J}) - \kappa \operatorname{tr}(\mathbf{J})^2\mbox{, with }\kappa = 0.05$$ |

In the cell below, write the function `orientation_features()` which implements the whole chain of processing of the algorithm described in the [flowchart](#Orientation-laboratory-(12-points)) at the start of this lab. Use the `structure_tensor` function you prepared in the previous section to get the structure tensor and use the relations shown in the table above to calculate the specified features (**1 point each**).
    
**Note:** For $\arctan\left(\frac{x1}{x2}\right)$ in $[-\pi, \pi]$ use `np.arctan2(x1, x2)`.
    
**Beware:** At each pixel where $J_{xx}[m,n]+J_{yy}[m,n] < 0.01$, set the **coherence to 0** to avoid a division by zero.

In [ ]:
# Function that calculates the orientation features of an image for a given sigma
def orientation_features(img, sigma):
    orientation = np.zeros(img.shape)
    energy      = np.zeros(img.shape)
    coherence   = np.zeros(img.shape)
    harris      = np.zeros(img.shape)
    
    # YOUR CODE HERE
    
    return orientation, energy, coherence, harris

In [ ]:
# Change the input image and sigma, to see the result on different images:
# fingerprint, wave_ramp, turbulence, traces or fourierhouse
input_img = fingerprint
sigma = 2
image_list = [input_img] + list(orientation_features(input_img, sigma=sigma))
# Display the output features
title_list = ['Input image', 'Orientation', 'Energy', 'coherence', 'Harris Index']
plt.close('all')
view = viewer(image_list, title=title_list, widgets=True)

As a sanity check, you can run the next four cells to check that each of the output features is in the correct range when applying the function to the `wave_ramp` image using `sigma = 2`. This does not mean that everything is correct but it can help to detect basic calculation errors.

In [ ]:
# Sanity checks
features = orientation_features(wave_ramp, sigma=2)
# Orientation
if (np.allclose(features[0], np.zeros(wave_ramp.shape))):
    print(f'WARNING!\nThe output orientation map of your function is all zeros, make sure your implementaion is not empty.')
elif not abs(features[0].min() + np.pi/2) < 0.01 and abs(features[0].max() - np.pi/2) < 0.01:
    print(f'WARNING!\nThe orientation should be in [-pi/2, pi/2], not in [{features[0].min():.3f}, {features[0].max():.3f}]')
else:
    print('The orientation is in the correct range.')

In [ ]:
# Sanity checks
features = orientation_features(wave_ramp, sigma=2)
# Energy
if (np.allclose(features[1], np.zeros(wave_ramp.shape))):
    print(f'WARNING!\nThe output energy map of your function is all zeros, make sure your implementaion is not empty.')
elif not abs(features[1].min() - 17) < 1 and abs(features[1].max() - 29) < 1:
    print(f'WARNING!\nFor this image, the energy should be in the range [~17, ~29], but it is in [{features[1].min():.3f}, {features[1].max():.3f}]')
else:
    print('The energy is in the correct range for this image.')

In [ ]:
# Sanity checks
features = orientation_features(wave_ramp, sigma=2)
# coherence
if (np.allclose(features[2], np.zeros(wave_ramp.shape))):
    print(f'WARNING!\nThe output coherence map of your function is all zeros, make sure your implementaion is not empty.')
elif not abs(features[2].min()) < 0.01 and abs(1 - features[2].max()) < 0.01:
    print(f'WARNING\nThe coherence should be in the range [0, 1], not in [{features[2].min():.3f}, {features[2].max():.3f}]')
else:
    print('The coherence is in the correct range.')

In [ ]:
# Sanity checks
features = orientation_features(wave_ramp, sigma=2)
# Harris index
if (np.allclose(features[3], np.zeros(wave_ramp.shape))):
    print(f'WARNING!\nThe output Harris index map of your function is all zeros, make sure your implementaion is not empty.')
elif not abs(features[3].min() + 41.5) < 1 and abs(features[3].max() - 111.3) < 1:
    print(f'WARNING!\nFor this image, the Harris index should be in the range [~-41.5, ~111.3], not in [{features[3].min():.3f}, {features[3].max():.3f}]')
else:
    print('The Harris index is in the correct range.')

### Feature visualization (2 points)

Until now, we have used grayscale (2D) images to visualize the orientation features. With grayscale, only one feature can be displayed. It is often convenient to visualize multiple features simultaneously and integrate them into the original image to make visual analysis more intuitive. One way to achieve this is to use the **hue, saturation, value (HSV)** color representation, which is sometimes also called HSB (for brightness), and is depicted in the figure below. Using this method, we can assign the orientation (a $2\pi$ periodic value) to the hue (which also lives in a circular scale), the coherence (a value between $0$ and $1$) to the saturation, and the brightness of the original image to the value, which will allow us to see the objects in the image. This will result on much clearer representations of the orientation features.

<center><img src="images/hsv_color_representation.png" alt="Drawing" style="width: 400px;"/></center>

In the cell below, **for 2 points**, implement the function `colorize_features` that takes as input parameters the *orientation*, the *coherence*, the *input image*, and a *mode* (of two possible modes, see table below). This function should
1. create an HSV image `hsv_image` from the orientation features, and
2. convert this HSV image to an RGB image using the function `cv.cvtColor` in order to display it with the `viewer`.
 
Below, both `hsv_img` and `rgb_img` are 3D arrays with dimensions `(height, width, channels)` where the $i$-th channel can be accessed as `img[:, :, i]` and it contains the information relative to the letter in position $i$ of the name of the respective color representation (HSV or RGB: $i=0$, hue or red channel, $i=1$, saturation or green channel, $i=2$, value or blue channel). 

This function will have two modes. Mode $0$ simply maps the orientation to the hue channel, while mode $1$ overlays the orientation and coherence features on top of the input image. See the table below for the specification of the two modes.  

| Mode | H channel, range `[0, 180]` | S channel, range `[0, 255]` | V channel, range `[0, 255]` |
| :-: | :-: | :-: | :-: |
| 0: Orientation only | orientation | constant image of value `255` | constant image of value `255` |
| 1: Features on image | orientation | coherence | input image |
    
**Note:** The color conversion using `cv.cvtColor` is already implemented. Since this function requires images of type `uint8`, we also take care of the rounding and casting beforehand. Please simply modify the predefined variable `hsv_img` to create the HSV image without applying any rounding or type conversion to it.
    
**Beware:** Think carefully about how each channel should (or should not) be shifted and/or normalized. Any two pixels with the same brightness, orientation, and coherence features in different images should look the same! Note that the orientation as you computed it above is in the range $[-\pi/2,\pi/2]$, and the coherence is in the range $[0,1]$. Assume the input image is in the range $[0,255]$.

**Note:** If you are curious about how the conversion from HSV to RGB works, you can check out the formula [here](https://en.wikipedia.org/wiki/HSL_and_HSV#HSV_to_RGB).

In [ ]:
# Function that returns a colorized rgb image depending on the orientation features
def colorize_features(orientation, coherence, img, mode):
    # Fill hsv_img[:,:,0] to set the hue, hsv_img[:,:,1] to set the saturation, and hsv_img[:,:,2] to set the value
    hsv_img = np.zeros((img.shape[0], img.shape[1], 3))
    rgb_img = np.zeros((img.shape[0], img.shape[1], 3))
    
    # YOUR CODE HERE
    
    # Convert HSV to RGB
    rgb_img = cv.cvtColor(np.round(hsv_img).astype(np.uint8), cv.COLOR_HSV2RGB)
    
    return rgb_img

Now run the next two cells for a quick test on your function. As usual, remember that these tests are not definitive and that they do not guarantee the full points. 

In [ ]:
# Sanity check for mode 0. First we define a few arrays for which we know how the output will look like.
orientation = np.array([[-np.pi/2, -np.pi/4, 0, np.pi/4, np.pi/2]])
coherence   = np.array([[0, 65./255, 130./255, 195./255, 255./255]])
img = np.array([[255, 195, 130, 65, 0]])
colorized_img = colorize_features(orientation, coherence, img, mode=0)
check_img = np.array([[[255, 0, 0], [127, 255, 0], [0, 255, 255],[128,   0, 255], [255,   0,   0]]], dtype=np.uint8)
if not np.allclose(colorized_img, check_img):
    print('WARNING!\nYour colorization function is not yet correct for mode 0. Check the comparison below:')
    plt.close('all')
    view = viewer([check_img, colorized_img], title=['Expected output', 'Your output'], subplots=(1,2))
else:
    print('Well done, your colorization function passed the sanity check for mode 0.')

In [ ]:
# Sanity check for mode 1
orientation = np.array([[-np.pi/2, -np.pi/4, 0, np.pi/4, np.pi/2]])
coherence   = np.array([[0, 65./255, 130./255, 195./255, 255./255]])
img = np.array([[255, 195, 130, 65, 0]])
colorized_img = colorize_features(orientation, coherence, img, mode=1)
check_img = np.array([[[255, 255, 255], [170, 195, 145], [64, 130, 130],[40,  15,  65], [0, 0, 0]]], dtype=np.uint8)
if not np.allclose(colorized_img, check_img):
    print('WARNING!\nYour colorization function is not yet correct for mode 1. Check the comparison image below:')
    plt.close('all')
    view = viewer([check_img, colorized_img], title=['Expected output', 'Your output'], subplots=(1,2))
else:
    print('Well done, your colorization function passed the sanity check for mode 1.')

Run the cell below to visualize the effect of the `colorize_features()` function on different images. 

**Note:** Click on `Extra Widgets` to change the mode and sigma values and apply the colorization by clicking on `Apply Colorization`. Cycle through the different images by clicking on `Next` and `Prev`.

In [ ]:
# Define control widgets for "Extra Widgets"
mode_dropdown = widgets.Dropdown(options=['0: Orientation only', '1: Features on image'],value='1: Features on image',description='Mode:',disabled=False)
mode_dictionary = {'0: Orientation only':0, '1: Features on image':1}
sigma_slider = widgets.IntSlider(value = 3, min = 1, max = 15, step = 1, description = r'$\sigma$')
button = widgets.Button(description = 'Apply Colorization')

def colorization_callback(img):
    mode = mode_dictionary[mode_dropdown.value]
    sigma = sigma_slider.value
    # Get the features using the function from part 2.1
    features = orientation_features(img, sigma=sigma)
    # Create the colorized image
    output = colorize_features(features[0], features[2], img, mode=mode)
    return output

plt.close('all')
image_list = [fingerprint, wave_ramp, turbulence, traces, fourierhouse]
new_widgets = [mode_dropdown, sigma_slider, button]
view = viewer(image_list, new_widgets=new_widgets, callbacks=[colorization_callback], widgets=True)

## Application

In this section, you will implement some applications that rely on the functions you implemented in the previous sections, in order to show you what they could be used for in real-life scenarios.

### Vector field (3 points)

In this exercise, we want to compute and visualize a vector field that represents the average directions of an image on a regular grid. To do this, we will first split the image into $m_y \times m_x$ blocks of size $N \times N$, and then we compute the average direction and coherence for each of those blocks, which will finally allow us to deduce the direction and length of the vectors.

<table><tr>
<td>
  <p align="center" style="padding: 0px">
    <img alt="Structuring elements" src="images/blocks_showcase.png" width="300"><br>
    <em style="color: grey">The image split into blocks of size N x N.</em>  
  </p>
</td>
<td>
  <p align="center" style="padding: 0px">
    <img alt="Structuring elements" src="images/turbulences_showcase.png" width="297"><br>
    <em style="color: grey">An example of how the final vector field should look like.</em>  
  </p>
</td>
</tr></table>

First of all, we need a function that divides the image into the mentioned blocks, so that we can treat each block separately. For this, we provide you with the function `create_blocks(img, N)`, which splits the `img` into blocks of size `N` $\times$ `N` and also returns the center location of each block, which we will later use as the location to draw the vectors. Run the cell below to define this function and make sure you understand its code.

**Note:** Make sure that you understand how we manage to center the blocks in the image in the (very common) case where the image size is not an integer multiple of the block size.

In [ ]:
# Function that divides the image into blocks of size N x N
def create_blocks(img, N):
    # Number of blocks in each direction
    my = img.shape[0] // N
    mx = img.shape[1] // N
    # Distance from the outer-most blocks to the edge of the image
    ry = (img.shape[0] % N) // 2
    rx = (img.shape[1] % N) // 2
    # Output variables
    out = np.zeros((my, mx, N, N))
    locs = np.zeros((my, mx, 2), dtype=int)
    # Split image into blocks and add their center locations
    for i in range(my):
        for j in range(mx):
            out[i, j] = img[ry + i*N:ry + (i+1)*N, rx + j*N:rx + (j+1)*N]
            locs[i, j] = [ry + i*N + N//2, rx + j*N + N//2]
    return out, locs

Now, **for 1 point**, implement the adapted structure tensor in the function `structure_tensor_blocks`. This adapted structure tensor has three main differences from the original structure tensor:
1. It operates on multiple blocks instead of a single image, this means the input variable is of size $(m_y, m_x, N, N)$.
2. The smoothing operation (`cv.GaussianBlur`) is replaced by a simple averaging operation ([`np.mean`](https://numpy.org/doc/stable/reference/generated/numpy.mean.html)).
3. The size of the structure tensor elements $J_{xx}$, $J_{yy}$ and $J_{xy}$ is not the same size as the original image, but instead is of size $(m_y, m_x)$.

In [ ]:
# Function that calculates the elements Jxx, Jxy and Jyy of the structure tensor matrix for each block by averaging
def structure_tensor_blocks(blocks):
    Jxx = np.zeros(blocks.shape[:2])
    Jxy = np.zeros(blocks.shape[:2])
    Jyy = np.zeros(blocks.shape[:2])
    
    for i in range(Jxx.shape[0]):
        for j in range(Jxx.shape[1]):
            # Generate gradientX and gradientY of the block at (i, j)
            
            # YOUR CODE HERE

            # Calculate fxx, fxy and fyy
            
            # YOUR CODE HERE

            # Average fxx, fxy and fyy to create Jxx[i, j], Jxy[i, j] and Jyy[i, j] for the block at (i, j)
            # YOUR CODE HERE
    
    return Jxx, Jxy, Jyy

To test your adapted structure tensor, we first create a test image of size $22 \times 22$ that consists of four $11 \times 11$ regions: a single point, a horizontal line, a vertical line and a diagonal line. Run the next cell to define and visualize this test image.

In [ ]:
test_img = np.zeros((22, 22), dtype=np.uint8)
# Point
test_img[5, 5] = 255
# Horizontal line
test_img[5, 11:] = 255
# Vertical line
test_img[11:, 5] = 255
# Diagonal line
test_img[list(range(11, 22)), list(range(11, 22))[::-1]] = 255
# Display test image
plt.close('all')
view = viewer(test_img)

Now we will apply your `structure_tensor_blocks` function to the test image by splitting it into the four previously mentioned $11 \times 11$ blocks. This means your resulting structure tensor elements $J_{xx}$, $J_{yy}$ and $J_{xy}$ should be of size $2 \times 2$. We will also test some specific properties that should correspond to these particular test blocks. You can read through the code below to see what we test exactly. Then run the cell below to perform the sanity check.

In [ ]:
# Create test blocks and locs
test_blocks, test_locs = create_blocks(test_img, 11)
# Run the function on the test images
Jxx, Jxy, Jyy = structure_tensor_blocks(test_blocks)
plt.close('all')
st_view = viewer([Jxx, Jyy, Jxy], subplots=(2, 2), cmap='viridis')

check_error = False
# Run sanity checks
if not (Jxx.shape == (2, 2)):
    print(f'WARNING!\nJxx should be a 2 x 2 array, yours has shape {Jxx.shape}'); check_error = True
if not (Jyy.shape == (2, 2)):
    print(f'WARNING!\nJyy should be a 2 x 2 array, yours has shape {Jyy.shape}'); check_error = True
if not (Jxy.shape == (2, 2)):
    print(f'WARNING!\nJxy should be a 2 x 2 array, yours has shape {Jxy.shape}'); check_error = True
if not (Jxx[0, 1] == 0):
    print(f'WARNING!\nJxx of the horizontal line should be 0, yours is {Jxx[0, 1]:.3f}'); check_error = True
if not (np.max(Jxx) == Jxx[1, 0] and Jxx[1, 0] != 0):
    print(f'WARNING!\nJxx should be maximal for the vertical line.'); check_error = True
if not (Jyy[1, 0] == 0):
    print(f'WARNING!\nJyy of the vertical line should be 0, yours is {Jyy[1, 0]:.3f}'); check_error = True
if not (np.max(Jyy) == Jyy[0, 1] and Jyy[0, 1] != 0):
    print(f'WARNING!\nJyy should be maximal for the horizontal line.'); check_error = True
if not (Jxy[0, 0] == 0 and Jxy[0, 1] == 0 and Jxy[1, 0] == 0 and Jxy[1, 1] != 0):
    print(f'WARNING!\nJxy should be zero for all test blocks, except for the diagonal line.'); check_error = True
if not check_error:
    print('Nice, your structure_tensor_blocks function passed the sanity check. Remember that this is not a definitive test!')

Next, we need a function that draws the vector lines on our image, according to the orientation and coherence value of each block. **For 1 point**, complete the function `draw_vector_field` that takes as input parameters:
* `img` : The image on which the vector field should be drawn.
* `locs` : The locations where the vector lines should be drawn. This is an array of size $(m_y, m_x, 2)$.
* `N`: The size of the blocks.
* `orientation` : The orientation values for each block. This is an array of size $(m_y, m_x)$.
* `coherence` : The coherence values for each block. This is an array of size $(m_y, m_x)$.

Most of this function's code is already implemented, the only part still missing is the calculation of the starting and end points of the vector lines. The vector lines for each block should be **centered at the location of their corresponding block**, **oriented according to the orientation of the corresponding block** and should be of **length** $l =$ coherence $ \cdot N$. The points $(x_0, y_0)$ and $(x_1, y_1)$, which you need to calculate, are the starting and end points of the line **relative to the center of their corresponding block**.

**Hints:**

- You may need to grab a pen and paper and do some simple geometry to obtain the correct results.
- Keep in mind that the y-axis in images is flipped compared to the Cartesian coordinate system.
- Thanks to the power of Python and NumPy, you can easily calculate the points for every block at once without needing any *for*-loops.
- You may want to use the functions [`np.sin`](https://numpy.org/doc/stable/reference/generated/numpy.sin.html) and [`np.cos`](https://numpy.org/doc/stable/reference/generated/numpy.cos.html).

In [ ]:
# Function that draws the vector field
def draw_vector_field(img, locs, N, orientation, coherence):
    # Convert to color image
    out = cv.cvtColor(img, cv.COLOR_GRAY2RGB)
    
    # Calculate the starting and end points of the lines according to the coherence and orientation
    # Note: x0, y0, x1 and y1 should be arrays of the same size as orientation and coherence!
    x0 = np.zeros(orientation.shape)
    y0 = np.zeros(orientation.shape)
    x1 = np.zeros(orientation.shape)
    y1 = np.zeros(orientation.shape)
    
    # YOUR CODE HERE
    
    # Draw the lines
    for i in range(locs.shape[0]):
        for j in range(locs.shape[1]):
            # Get the maximum range of values
            rnge = int(max(np.abs(x1[i, j]-x0[i, j]), np.abs(y1[i, j]-y0[i, j])))
            # Get line indices
            x = locs[i, j, 1] + np.linspace(x0[i, j], x1[i, j], rnge+1).astype(int)
            y = locs[i, j, 0] + np.linspace(y0[i, j], y1[i, j], rnge+1).astype(int)
            # Clipping mask to prevent drawing outside the image (rare case)
            mask = np.bitwise_and(y < out.shape[0], x < out.shape[1])
            # Set the image at the line indices to red
            out[y[mask], x[mask]] = np.array([255, 0, 0])
    return out

In the next cell, we will again run a simple sanity check on your function, using the same test image from before.

In [ ]:
# Create the test orientation and coherence arrays
test_orientation = np.array([[0, 0], [np.pi/2, np.pi/4]])
test_coherence = np.array([[0, 1], [1, 1]])
# Apply function
test_vector_field = draw_vector_field(test_img, test_locs, 11, test_orientation, test_coherence)

# Create correct image
red_mask = test_img == 255
red_mask[np.array([21, 20, 12, 11]), np.array([11, 12, 20, 21])] = False
correct_img = cv.cvtColor(test_img, cv.COLOR_GRAY2RGB); correct_img[red_mask] = [255, 0, 0]

plt.close('all')
if not np.allclose(test_vector_field, correct_img):
    print('WARNING!\nYour draw_vector_field function is not yet correct. Check the images below to see what is wrong.')
    view = viewer([test_vector_field, correct_img], title=['Your vector field', 'Correct vector field'], subplots=(1,2))
else:
    print('Good job, your draw_vector_field function produces the correct result on the test image.')
    view = viewer(test_vector_field, title='Your vector field')

The final step of this exercise is to implement the complete vector field pipeline in the function `generate_vector_field`. This function will simply take an image `img` and the block size `N`, and return the input image with the vector field drawn on it. The only part that is missing from this function is the $\operatorname{orientation}$ and $\operatorname{coherence}$ calculation, which is exactly the same as you coded for the `structure_tensor` function. So, **for 1 point**, complete the function `generate_vector_field` in the cell below, by adding the $\operatorname{orientation}$ and $\operatorname{coherence}$ calculation.

In [ ]:
# Function that calculates the orientation features of an image for a given sigma
def generate_vector_field(img, N):
    # Cut image into blocks
    blocks, locs = create_blocks(img, N)
    
    orientation = np.zeros(blocks.shape[:2])
    coherence   = np.zeros(blocks.shape[:2])
    
    # Get components of Structure tensor corresponding to each block
    Jxx, Jxy, Jyy = structure_tensor_blocks(blocks)
    
    # Calculate orientation and coherence
    
    # YOUR CODE HERE
    
    # Draw the vector field
    out = draw_vector_field(img, locs, N, orientation, coherence)
    
    return out

In the cell below, you can use the interactive viewer to play around with different block sizes and see the effect on different images. Run the cell and click on <code>Extra Widgets</code> to get access to the controls.

In [ ]:
# Define control widgets for "Extra Widgets"
N_slider = widgets.IntSlider(value=40, min=10, max=256, step=1, description='N')
button = widgets.Button(description='Generate vector field')

def vector_field_callback(img):
    return generate_vector_field(img, N_slider.value)

plt.close('all')
image_list = [turbulence, wave_ramp, fingerprint, traces, fourierhouse]
view = viewer(image_list, new_widgets=[N_slider, button], callbacks=[vector_field_callback], widgets=True)

### Advanced: Isotropic filtering in the Fourier space (1 point)

In the first exercise, you were told to use the Sobel filter to compute the gradient for the structure tensor. As you have seen, this has worked well for everything we have done until now. However, if we inspect the resulting orientation image more closely, we can observe some anomalous behavior. To illustrate this, let us first create a perfect test image that contains (inside a certain radius of the image) exactly the same number of pixels for every possible orientation.

Run the next cell to create this test image and define two useful functions that we will need later.

In [ ]:
# Function that creates the orientation test image. This image is perfectly isotropic
def create_test_img(ny, nx):
    # Minimum and maximum radial frequencies
    fmin = 0.02; fmax = 8 * fmin;
    # Center
    hx = nx / 2; hy = ny / 2;
    n = min(nx, ny);
    def structure_func(j, i):
        # Radius
        r = np.sqrt((i - hx)**2 + (j - hy)**2);
        # Radial envelope
        u = 1.0 / (1.0 + np.exp((r - n * 0.45) / 2.0));
        # Radial frequency profile
        f = fmin + r * (fmax - fmin) / n;
        # Radial modulating function
        v = np.sin(np.pi * 2 * f * r);
        return (1.0 + v * u) * 128
    return np.fromfunction(structure_func, shape=(ny, nx)).astype(np.uint8)

# Calculates the Fourier Transform
def get_FT(img):
    return np.fft.fftshift(np.fft.fft2(img))

# Calculates the inverse Fourier Transform
def get_iFT(img):
    return np.fft.ifft2(np.fft.ifftshift(img)).real

test_img = create_test_img(2048, 2048)
plt.close('all')
view = viewer(test_img)

Now, in theory, if we extract the orientations of a circular cut-out of this test image using the `orientation_features` function that you coded in [Part 2.A.]((#2.A.-Feature-calculation-(4-points))) and plot the distribution of these orientations, we should get a completely flat line. Let's see how it actually looks: run the cell below to display the orientation image and its distribution inside a circular cut-out.

In [ ]:
# Get the old orientation
orientation = orientation_features(test_img, sigma=5)[0]  
# Generate the disk-shaped mask to only analize the colors inside the correct radius
mask = np.fromfunction(lambda i, j: np.sqrt((i-orientation.shape[0]//2)**2 + (j-orientation.shape[1]//2)**2), 
                       shape=orientation.shape)    
r = np.min(mask.shape)//2 - np.min(mask.shape)//20
# Generate histograms using the mask
hist, edg = np.histogram(orientation[mask < r], bins = 1000, range = (orientation.min(), orientation.max()))
cent = (edg[0:-1] + edg[1:])/2
# Display the masked orientation
orientation[mask >= r] = -np.pi/2
plt.close('all')
view = viewer([orientation], title=['Orientation of the test image'], cmap='hsv')
# Display the angle distribution
f = plt.figure(); plt.title('Orientation distribution'); plt.xlabel('Angle [rad]'); plt.ylabel('# pixels')
plt.plot(cent, hist); plt.grid(); 
plt.xticks(ticks=[-np.pi/2,-np.pi/4,0,np.pi/4,np.pi/2],labels=[r'$-\pi/2$',r'$-\pi/4$',r'$0$',r'$\pi/4$',r'$\pi/2$']); plt.show()

In the first image, you will probably not see any problem since it's very hard to see it by just looking at the orientation image. However, in the distribution plot you can clearly see that some orientations are preferred over others. Specifically, ignoring the very sharp peaks at $0$, $\pm\frac{\pi}{4}$ and $\pm\frac{\pi}{2}$ (an artefact of having a limited number of pixels), there are more pixels with an orientation close to $0$ or $\pm\frac{\pi}{2}$ and less pixels close to $\pm\frac{\pi}{4}$. 

Now why could that be? If we look more closely at the Sobel filter masks $h_x$ and $h_y$

$$h_x = 
\begin{bmatrix} 
    1 & 0 & -1 \\
    2 & 0 & -2 \\ 
    1 & 0 & -1 
\end{bmatrix}
,\;\;\;\;
h_y = 
\begin{bmatrix} 
    1 & 2 & 1 \\
    0 & 0 & 0 \\ 
    -1 & -2 & -1 
\end{bmatrix}
$$

we can see that this filter is not isotropic, meaning it doesn't treat all directions equally. In this case, as we saw in the orientation distribution above, the horizontal and vertical edges are favored, resulting in a biased orientation distribution.

In order to have a non-biased orientation detector, we need to re-implement the `structure_tensor` function using an isotropic gradient filter. There are several to choose from but in this exercise we will calculate the gradient using the Fourier property

$$\frac{\partial f(x,y)}{\partial x} \xrightarrow{\mathcal{F}} j\omega_x\operatorname{F}(\omega_x, \omega_y), \;\;\; 
\frac{\partial f(x,y)}{\partial y} \xrightarrow{\mathcal{F}} j\omega_y\operatorname{F}(\omega_x, \omega_y)\,.$$

As you can see we only need to take the Fourier transform of our image and multiply it with either $j\omega_x$ or $j\omega_y$. Then we can take the inverse Fourier transform and we will have our derivatives in the $x$ and $y$ directions.

**For 1 point**, implement the functions `get_w` and `structure_tensor_improved` in the cells below.

The first step for you is to implement the function `get_w` that simply returns a vector $\omega$ consisting of $m$ equidistant points going from $-\frac{\pi}{2}$ to $\frac{\pi}{2}$.

This is a very simple function that will be used in `structure_tensor_improved` to generate the complex-valued vectors $j\omega_x$ and $j\omega_y$.

**Hint:** Use the function [`np.linspace`](https://numpy.org/doc/stable/reference/generated/numpy.linspace.html).

In [ ]:
# Calculates omega (w) of length m
def get_w(m):
    w = np.zeros(m)
    
    # YOUR CODE HERE
    
    return w

# Visualize omega
plt.close('all')
plt.figure("Frequency vector")
plt.plot(get_w(100))
plt.ylabel(r'$\omega$'); plt.xlabel("Vector index"); plt.grid()
plt.yticks(ticks=[-np.pi/2,-np.pi/4,0,np.pi/4,np.pi/2],labels=[r'$-\pi/2$',r'$-\pi/4$',r'$0$',r'$\pi/4$',r'$\pi/2$']);
plt.show()

Run the next cell for a quick test on $\omega$.

In [ ]:
# Perform test on a vector of 10 elements. We will check the length, range, and a direct comparison with the solution
w_out = get_w(10)
check = True
if not len(w_out) == 10:
    check = False
    print(f'WARNING!\nThe length of the output vector should be equal to the input parameter m. Expected length=10, your length={len(w_out)}.')
if not np.min(w_out) == -np.pi/2 and np.max(w_out) == np.pi/2:
    check = False
    print(f'WARNING!\nOmega should go from -1.57079633 to 1.57079633 not from {np.min(w_out):.8f} to {np.max(w_out):.8f}!')
if not np.allclose(w_out, [-1.57079633, -1.22173048, -0.87266463, -0.52359878, -0.17453293, 
                           0.17453293,  0.52359878,  0.87266463,  1.22173048,  1.57079633]):
    check = False
    print('WARNING!\nYour omega vector is not yet correct!')
if check:
    print('Well done, the function passed the sanity check.')

In the next cell, complete the function `structure_tensor_improved` by implementing the Fourier domain gradient filters as explained above. The code that generates the arrays $j\omega_x$ and $j\omega_y$ is already provided. It uses your function `get_w` to fill both the rows of the imaginary part of `jw_x` and the columns of the imaginary part of `jw_y` with the vector $\omega$.

**Hint:** You can use the functions `get_FT(img)` to get the complex Fourier transform of `img` and `get_iFT(img_FT)` to get the inverse Fourier transform of `img_FT`. Those functions also take care of performing the correct shifting.

**Hint:** Implementing these filters might be easier than you think! Or exactly as easy as you thought, but did not know why. Have a look at the NumPy broadcasting rules [here](https://numpy.org/doc/stable/user/basics.broadcasting.html).

In [ ]:
# Function that calculates the elements Jxx, Jxy and Jyy of the structure tensor matrix
def structure_tensor_improved(img, sigma):
    Jxx = None
    Jxy = None
    Jyy = None
    
    # Generate the complex-numbered filters jw_x and jw_y
    jw_x = 1j*get_w(img.shape[1])
    jw_y = 1j*get_w(img.shape[0]).reshape((img.shape[0], 1))
    
    # Gradient calculation using jw_x and jw_y in the Fourier space
    # YOUR CODE HERE
    
    # Calculate fxx, fxy, fyy and then Jxx, Jxy and Jyy from the gradients
    # in exactly the same way as in the original structure_tensor function (Part 1)
    # YOUR CODE HERE
    
    return Jxx, Jxy, Jyy

In this part you will need to do the sanity check of the `structure_tensor_improved` function by eye. The cell below will run the function `orientation_features` twice on the `test_img`, once with the old `structure_tensor` and once with the improved version. 

You will be able to see the results in a <code>viewer</code>, where you will see the orientation feature for each version of the `structure_tensor` function, as well as the difference between both results. Once you are sure on the correctness of `structure_tensor_improved`, reflect on the difference and its origins. Moreover, you will see a plot of both orientation distributions. 

The displayed orientation feature is for you to see that you did not mess up the function, and you will most likely not be able to see much difference between the two versions. However, in the orientation distribution plot you should be able to clearly distinguish the Fourier version from the Sobel one. **The Fourier version should be completely flat (with exception of a few sharp spikes)** compared to the Sobel version that looks more like a sinusoidal wave.

In [ ]:
# Get the old orientation
orientation = orientation_features(test_img, sigma=5)[0]
# Replace the structure tensor function by the improved one
old_ST = structure_tensor
structure_tensor = structure_tensor_improved
# Get the improved orientation
try:
    orientation_improved = orientation_features(test_img, sigma=5)[0]
except:
    # Handle errors in the code
    structure_tensor = old_ST
    raise
# Restore the old structure tensor function
structure_tensor = old_ST    

# Generate the disk-shaped mask to only analize the colors inside the correct radius
mask = np.fromfunction(lambda i, j: np.sqrt((i-orientation.shape[0]//2)**2 + (j-orientation.shape[1]//2)**2), 
                       shape=orientation.shape)    
r = np.min(mask.shape)//2-np.min(mask.shape)//20

# Generate histograms using the mask
hist, edg = np.histogram(orientation[mask < r], bins = 1000, range = (orientation.min(), orientation.max()))
hist_improved, edg_improved = np.histogram(orientation_improved[mask < r], bins = 1000, 
                                range = (orientation_improved.min(), orientation_improved.max()))
cent = (edg[0:-1] + edg[1:])/2;
cent_improved = (edg_improved[0:-1] + edg_improved[1:])/2;
# Display the two masked orientations
orientation[mask >= r] = -np.pi/2
orientation_improved[mask >= r] = -np.pi/2
plt.close('all')
image_list = [orientation, orientation_improved]
title_list = ['Sobel orientation', 'Fourier orientation']
view = viewer(image_list, title=title_list, cmap='hsv', widgets=True)

# Display the angle distribution
plt.figure()
plt.title('Orientation distribution'); plt.xlabel('Angle [rad]'); plt.ylabel('# pixels')
plt.xticks(ticks=[-np.pi/2,-np.pi/4,0,np.pi/4,np.pi/2],labels=[r'$-\pi/2$',r'$-\pi/4$',r'$0$',r'$\pi/4$',r'$\pi/2$']);
plt.plot(cent, hist)
plt.plot(cent_improved, hist_improved)
plt.legend(['Sobel orientation', 'Fourier orientation']); plt.grid(); plt.show()

🎉 Congratulations on finishing the second part of the orientation lab!

Make sure to save your notebook (you might want to keep a copy on your personal computer) and upload it to Moodle, **in a zip file with the other notebook of this lab.**

* Keep the name of the notebook as: *2_orientation.ipynb*,
* Name the `zip` file: *orientation_lab.zip*.